In [1]:
# ==================================================
# 1. PROJECT SETUP AND CENTRALIZED CONFIGURATION
# ==================================================

import sys
from pathlib import Path
import json
from collections import Counter

import torch
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# --------------------------------------------------
# Locate project root
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

while not (
    (PROJECT_ROOT / "01_src" / "config" / "pipeline_config.py").exists()
    and (PROJECT_ROOT / "02_processed_data").exists()
):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")

    PROJECT_ROOT = PROJECT_ROOT.parent

# --------------------------------------------------
# Import centralized configuration
# --------------------------------------------------

SRC_ROOT = PROJECT_ROOT / "01_src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from config.pipeline_config import (
    PROCESSED_DATA,
    TRAINING_SIZE,
    VALIDATION_SIZE,
    TEST_SIZE,
    CLIP_MODEL_NAME,
    SBERT_MODEL_NAME,
    CLIP_BATCH_SIZE,
    SBERT_BATCH_SIZE,
)

# --------------------------------------------------
# Define common split configuration
# --------------------------------------------------

DATA_ROOT = PROCESSED_DATA

SPLIT_SIZES = {
    "training": TRAINING_SIZE,
    "validation": VALIDATION_SIZE,
    "test": TEST_SIZE,
}

SPLITS = tuple(SPLIT_SIZES.keys())

# --------------------------------------------------
# Display configuration
# --------------------------------------------------

print("Project root:", PROJECT_ROOT)
print("Processed data:", DATA_ROOT)
print("CLIP model:", CLIP_MODEL_NAME)
print("SBERT model:", SBERT_MODEL_NAME)
print("Split sizes:", SPLIT_SIZES)

Project root: c:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG
Processed data: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data
CLIP model: openai/clip-vit-base-patch32
SBERT model: sentence-transformers/all-MiniLM-L6-v2
Split sizes: {'training': 9900, 'validation': 100, 'test': 100}


In [2]:
# ==================================================
# 2. INSPECT PROCESSED DATASET STRUCTURE
# ==================================================

ANNOTATION_FILES = [
    "captions.json",
    "instances.json",
    "discarded_captions.json",
]

for split in SPLITS:

    split_dir = DATA_ROOT / split
    images_dir = split_dir / "images"
    annotations_dir = split_dir / "annotations"

    print(f"\n{'=' * 45}")
    print(f" {split.upper()}")
    print(f"{'=' * 45}")

    # --------------------------------------------------
    # Check directories
    # --------------------------------------------------

    assert split_dir.exists(), f"Missing split directory: {split_dir}"
    assert images_dir.exists(), f"Missing images directory: {images_dir}"
    assert annotations_dir.exists(), (
        f"Missing annotations directory: {annotations_dir}"
    )

    # --------------------------------------------------
    # Count images
    # --------------------------------------------------

    image_files = list(images_dir.glob("*.jpg"))
    expected_images = SPLIT_SIZES[split]

    print("Expected images:", expected_images)
    print("Found images:", len(image_files))

    assert len(image_files) == expected_images, (
        f"{split}: expected {expected_images} images, "
        f"but found {len(image_files)}"
    )

    # --------------------------------------------------
    # Inspect annotation files
    # --------------------------------------------------

    for filename in ANNOTATION_FILES:

        file_path = annotations_dir / filename

        if not file_path.exists():
            print(f"\n{filename}: NOT FOUND")
            continue

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print(f"\n{filename}")
        print("Top-level keys:", list(data.keys()))

        if "annotations" in data:
            print("Number of annotations:", len(data["annotations"]))

        if "images" in data:
            print("Number of image records:", len(data["images"]))

print("\nDataset structure inspection completed successfully.")


 TRAINING
Expected images: 9900
Found images: 9900

captions.json
Top-level keys: ['info', 'licenses', 'images', 'annotations']
Number of annotations: 49500
Number of image records: 9900

instances.json
Top-level keys: ['info', 'licenses', 'images', 'annotations', 'categories']
Number of annotations: 71326
Number of image records: 9900

discarded_captions.json
Top-level keys: ['split', 'selection_rule', 'discarded_caption_count', 'discarded_annotations']

 VALIDATION
Expected images: 100
Found images: 100

captions.json
Top-level keys: ['info', 'licenses', 'images', 'annotations']
Number of annotations: 500
Number of image records: 100

instances.json
Top-level keys: ['info', 'licenses', 'images', 'annotations', 'categories']
Number of annotations: 769
Number of image records: 100

discarded_captions.json
Top-level keys: ['split', 'selection_rule', 'discarded_caption_count', 'discarded_annotations']

 TEST
Expected images: 100
Found images: 100

captions.json
Top-level keys: ['info', 

In [3]:
# ==================================================
# 3. IMAGE INTEGRITY AND CORRUPTION CHECK
# ==================================================

image_integrity_results = {}

for split in SPLITS:

    images_dir = DATA_ROOT / split / "images"
    image_files = sorted(images_dir.glob("*.jpg"))

    readable = 0
    corrupted = []

    # Check each image
    for image_path in tqdm(
        image_files,
        desc=f"Checking {split} images"
    ):
        try:
            with Image.open(image_path) as img:
                img.verify()

            readable += 1

        except Exception as error:
            corrupted.append({
                "filename": image_path.name,
                "error": str(error)
            })

    # Store results
    image_integrity_results[split] = {
        "expected": SPLIT_SIZES[split],
        "total": len(image_files),
        "readable": readable,
        "corrupted": len(corrupted),
        "corrupted_examples": corrupted[:5]
    }

    # Display results
    print(f"\n{'=' * 45}")
    print(split.upper())
    print(f"{'=' * 45}")

    print("Expected images:", SPLIT_SIZES[split])
    print("Total images:", len(image_files))
    print("Readable images:", readable)
    print("Corrupted images:", len(corrupted))

    if corrupted:
        print("Examples:", corrupted[:5])

    # Validate image count
    assert len(image_files) == SPLIT_SIZES[split], (
        f"{split}: expected {SPLIT_SIZES[split]} images, "
        f"but found {len(image_files)}"
    )

# Final summary
total_images = sum(
    result["total"]
    for result in image_integrity_results.values()
)

total_corrupted = sum(
    result["corrupted"]
    for result in image_integrity_results.values()
)

print("\n" + "=" * 45)
print("IMAGE INTEGRITY SUMMARY")
print("=" * 45)
print("Total images checked:", total_images)
print("Total corrupted images:", total_corrupted)

print("\nImage integrity check completed.")

Checking training images:   0%|          | 0/9900 [00:00<?, ?it/s]


TRAINING
Expected images: 9900
Total images: 9900
Readable images: 9900
Corrupted images: 0


Checking validation images:   0%|          | 0/100 [00:00<?, ?it/s]


VALIDATION
Expected images: 100
Total images: 100
Readable images: 100
Corrupted images: 0


Checking test images:   0%|          | 0/100 [00:00<?, ?it/s]


TEST
Expected images: 100
Total images: 100
Readable images: 100
Corrupted images: 0

IMAGE INTEGRITY SUMMARY
Total images checked: 10100
Total corrupted images: 0

Image integrity check completed.


In [4]:
# ==================================================
# 4. IMAGE DIMENSIONS, COLOUR MODES AND EXIF ANALYSIS
# ==================================================

image_properties_results = {}

for split in SPLITS:

    images_dir = DATA_ROOT / split / "images"
    image_files = sorted(images_dir.glob("*.jpg"))

    dimensions = Counter()
    modes = Counter()
    exif_orientation = Counter()

    widths = []
    heights = []

    for image_path in tqdm(
        image_files,
        desc=f"Analysing {split} images"
    ):
        with Image.open(image_path) as img:

            width, height = img.size

            dimensions[(width, height)] += 1
            modes[img.mode] += 1

            widths.append(width)
            heights.append(height)

            # EXIF tag 274 represents image orientation.
            # Value 1 means normal orientation.
            orientation = img.getexif().get(274, 1)
            exif_orientation[orientation] += 1

    # Store results for later reporting
    image_properties_results[split] = {
        "image_count": len(image_files),
        "colour_modes": dict(modes),
        "common_dimensions": dimensions.most_common(10),
        "min_width": min(widths) if widths else None,
        "max_width": max(widths) if widths else None,
        "min_height": min(heights) if heights else None,
        "max_height": max(heights) if heights else None,
        "exif_orientations": dict(exif_orientation),
    }

    # Display results
    print(f"\n{'=' * 45}")
    print(split.upper())
    print(f"{'=' * 45}")

    print("Expected images:", SPLIT_SIZES[split])
    print("Image count:", len(image_files))
    print("Colour modes:", modes)
    print("Most common dimensions:", dimensions.most_common(10))

    if widths and heights:
        print("Width range:", min(widths), "-", max(widths))
        print("Height range:", min(heights), "-", max(heights))

    print("EXIF orientations:", exif_orientation)

    # Validate image count
    assert len(image_files) == SPLIT_SIZES[split], (
        f"{split}: expected {SPLIT_SIZES[split]} images, "
        f"but found {len(image_files)}"
    )

print("\nImage properties analysis completed.")

Analysing training images:   0%|          | 0/9900 [00:00<?, ?it/s]


TRAINING
Expected images: 9900
Image count: 9900
Colour modes: Counter({'RGB': 9881, 'L': 19})
Most common dimensions: [((640, 480), 2092), ((640, 427), 1185), ((480, 640), 742), ((640, 426), 450), ((500, 375), 396), ((427, 640), 338), ((640, 428), 287), ((640, 425), 274), ((612, 612), 196), ((375, 500), 173)]
Width range: 152 - 640
Height range: 109 - 640
EXIF orientations: Counter({1: 9900})


Analysing validation images:   0%|          | 0/100 [00:00<?, ?it/s]


VALIDATION
Expected images: 100
Image count: 100
Colour modes: Counter({'RGB': 99, 'L': 1})
Most common dimensions: [((640, 480), 16), ((640, 427), 14), ((480, 640), 5), ((640, 426), 5), ((500, 375), 5), ((640, 428), 4), ((427, 640), 4), ((375, 500), 4), ((640, 478), 3), ((640, 425), 2)]
Width range: 299 - 640
Height range: 238 - 640
EXIF orientations: Counter({1: 100})


Analysing test images:   0%|          | 0/100 [00:00<?, ?it/s]


TEST
Expected images: 100
Image count: 100
Colour modes: Counter({'RGB': 100})
Most common dimensions: [((640, 480), 22), ((500, 375), 8), ((640, 427), 7), ((640, 425), 7), ((426, 640), 5), ((640, 428), 4), ((640, 426), 4), ((427, 640), 3), ((480, 640), 3), ((640, 434), 2)]
Width range: 256 - 640
Height range: 200 - 640
EXIF orientations: Counter({1: 100})

Image properties analysis completed.


In [4]:
#%pip install transformers torch pillow

In [5]:
# ==================================================
# 5. LOAD CLIP IMAGE PROCESSOR
# ==================================================

from transformers import CLIPProcessor

clip_processor = CLIPProcessor.from_pretrained(
    CLIP_MODEL_NAME
)

print("CLIP processor loaded successfully.")
print("Model:", CLIP_MODEL_NAME)

CLIP processor loaded successfully.
Model: openai/clip-vit-base-patch32


In [6]:
# ==================================================
# 6. TEST IMAGE PREPROCESSING ON A SAMPLE IMAGE
# ==================================================

from data.preprocessing.image_preprocessing import (
    load_and_prepare_image,
    preprocess_image_for_clip
)

# Select one image from the training split
sample_split = "training"

sample_images_dir = DATA_ROOT / sample_split / "images"
sample_image_files = sorted(sample_images_dir.glob("*.jpg"))

assert sample_image_files, (
    f"No images found in {sample_images_dir}"
)

sample_image_path = sample_image_files[0]

# Load and prepare the original image
prepared_image = load_and_prepare_image(sample_image_path)

# Apply CLIP preprocessing
pixel_values = preprocess_image_for_clip(
    sample_image_path,
    clip_processor
)

# Display results
print("Original image:", sample_image_path.name)
print("Prepared image mode:", prepared_image.mode)
print("Prepared image dimensions:", prepared_image.size)
print("CLIP tensor shape:", pixel_values.shape)
print("Tensor data type:", pixel_values.dtype)

# Basic validation
assert prepared_image.mode == "RGB"
assert pixel_values.ndim == 4
assert pixel_values.shape[0] == 1
assert torch.isfinite(pixel_values).all()

print("\nSample image preprocessing passed successfully.")

Original image: 000000000149.jpg
Prepared image mode: RGB
Prepared image dimensions: (640, 428)
CLIP tensor shape: torch.Size([1, 3, 224, 224])
Tensor data type: torch.float32

Sample image preprocessing passed successfully.


In [7]:
# ==================================================
# 7. VALIDATE CLIP PREPROCESSING ACROSS ALL SPLITS
# ==================================================

# Get expected dimensions from the CLIP processor
crop_size = clip_processor.image_processor.crop_size

expected_height = crop_size["height"]
expected_width = crop_size["width"]

clip_preprocessing_results = {}

for split in SPLITS:

    images_dir = DATA_ROOT / split / "images"
    image_files = sorted(images_dir.glob("*.jpg"))

    successful = 0
    failed = []

    for image_path in tqdm(
        image_files,
        desc=f"Checking {split}"
    ):

        try:
            pixel_values = preprocess_image_for_clip(
                image_path,
                clip_processor
            )

            # Validate tensor shape
            expected_shape = (
                1, 3, expected_height, expected_width
            )

            assert tuple(pixel_values.shape) == expected_shape, (
                f"Unexpected tensor shape: {pixel_values.shape}"
            )

            # Validate tensor data type
            assert pixel_values.dtype == torch.float32, (
                f"Unexpected data type: {pixel_values.dtype}"
            )

            # Validate numerical values
            assert torch.isfinite(pixel_values).all(), (
                "Tensor contains NaN or infinite values."
            )

            successful += 1

        except Exception as error:
            failed.append({
                "filename": image_path.name,
                "error": str(error)
            })

    # Store results
    clip_preprocessing_results[split] = {
        "expected": SPLIT_SIZES[split],
        "total": len(image_files),
        "successful": successful,
        "failed": len(failed),
        "failure_examples": failed[:5],
        "tensor_shape": [
            1, 3, expected_height, expected_width
        ],
        "dtype": "float32"
    }

    # Display results
    print(f"\n{'=' * 45}")
    print(split.upper())
    print(f"{'=' * 45}")

    print("Expected images:", SPLIT_SIZES[split])
    print("Total images:", len(image_files))
    print("Successfully preprocessed:", successful)
    print("Failed:", len(failed))

    if failed:
        print("Examples of failures:", failed[:5])

    # Validate split size
    assert len(image_files) == SPLIT_SIZES[split], (
        f"{split}: expected {SPLIT_SIZES[split]} images, "
        f"but found {len(image_files)}"
    )

# Final summary
total_checked = sum(
    result["total"]
    for result in clip_preprocessing_results.values()
)

total_successful = sum(
    result["successful"]
    for result in clip_preprocessing_results.values()
)

total_failed = sum(
    result["failed"]
    for result in clip_preprocessing_results.values()
)

print("\n" + "=" * 45)
print("CLIP PREPROCESSING SUMMARY")
print("=" * 45)

print("Total images checked:", total_checked)
print("Successfully preprocessed:", total_successful)
print("Failed:", total_failed)
print("Expected tensor shape:", (
    1, 3, expected_height, expected_width
))

assert total_failed == 0, (
    f"CLIP preprocessing failed for {total_failed} images."
)

assert total_successful == sum(SPLIT_SIZES.values()), (
    "The number of successfully preprocessed images "
    "does not match the expected total."
)

print("\nAll CLIP preprocessing checks passed successfully.")

Checking training:   0%|          | 0/9900 [00:00<?, ?it/s]


TRAINING
Expected images: 9900
Total images: 9900
Successfully preprocessed: 9900
Failed: 0


Checking validation:   0%|          | 0/100 [00:00<?, ?it/s]


VALIDATION
Expected images: 100
Total images: 100
Successfully preprocessed: 100
Failed: 0


Checking test:   0%|          | 0/100 [00:00<?, ?it/s]


TEST
Expected images: 100
Total images: 100
Successfully preprocessed: 100
Failed: 0

CLIP PREPROCESSING SUMMARY
Total images checked: 10100
Successfully preprocessed: 10100
Failed: 0
Expected tensor shape: (1, 3, 224, 224)

All CLIP preprocessing checks passed successfully.


In [8]:
# ==================================================
# 8. SAVE IMAGE PREPROCESSING VALIDATION REPORT
# ==================================================

# Get tensor information from the validation results
first_split = SPLITS[0]

tensor_shape = clip_preprocessing_results[first_split]["tensor_shape"]
tensor_dtype = clip_preprocessing_results[first_split]["dtype"]

# Build the validation report
validation_results = {
    "stage": "image_preprocessing",
    "model": CLIP_MODEL_NAME,
    "input_format": "RGB",
    "tensor_shape": tensor_shape,
    "tensor_dtype": tensor_dtype,

    "results": {
        split: {
            "expected": clip_preprocessing_results[split]["expected"],
            "total": clip_preprocessing_results[split]["total"],
            "successful": clip_preprocessing_results[split]["successful"],
            "failed": clip_preprocessing_results[split]["failed"]
        }
        for split in SPLITS
    },

    "summary": {
        "total_images": sum(
            clip_preprocessing_results[split]["total"]
            for split in SPLITS
        ),
        "successful": sum(
            clip_preprocessing_results[split]["successful"]
            for split in SPLITS
        ),
        "failed": sum(
            clip_preprocessing_results[split]["failed"]
            for split in SPLITS
        )
    }
}

# Determine the actual validation status
validation_results["status"] = (
    "passed"
    if (
        validation_results["summary"]["failed"] == 0
        and all(
            validation_results["results"][split]["total"]
            == validation_results["results"][split]["expected"]
            for split in SPLITS
        )
    )
    else "failed"
)

# Save report
output_path = DATA_ROOT / "preprocessing_validation.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(validation_results, f, indent=4)

print("Validation results saved to:", output_path)
print("Validation status:", validation_results["status"])
print("Total images:", validation_results["summary"]["total_images"])
print("Successfully preprocessed:", validation_results["summary"]["successful"])
print("Failed:", validation_results["summary"]["failed"])

Validation results saved to: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\preprocessing_validation.json
Validation status: passed
Total images: 10100
Successfully preprocessed: 10100
Failed: 0


In [9]:
# ==================================================
# 9. INSPECT TRAINING CAPTION ANNOTATIONS
# ==================================================

# Select the split to inspect
caption_split = "training"

captions_path = (
    DATA_ROOT / caption_split / "annotations" / "captions.json"
)

# Verify that the annotation file exists
assert captions_path.exists(), (
    f"Caption annotation file not found: {captions_path}"
)

# Load caption annotations
with open(captions_path, "r", encoding="utf-8") as f:
    captions_data = json.load(f)

# Validate the annotation structure
required_keys = {"images", "annotations"}

assert required_keys.issubset(captions_data.keys()), (
    f"Missing required keys: {required_keys - captions_data.keys()}"
)

# Inspect the structure
print("Split:", caption_split.upper())
print("Top-level keys:", list(captions_data.keys()))

print("\nNumber of image records:",
      len(captions_data["images"]))

print("Number of caption annotations:",
      len(captions_data["annotations"]))

# Inspect one image record
print("\nExample image record:")
print(captions_data["images"][0])

# Inspect three caption annotations
print("\nExample captions:")

for annotation in captions_data["annotations"][:3]:
    print(annotation)

print("\nCaption annotation inspection completed.")

Split: TRAINING
Top-level keys: ['info', 'licenses', 'images', 'annotations']

Number of image records: 9900
Number of caption annotations: 49500

Example image record:
{'license': 3, 'file_name': '000000184613.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000184613.jpg', 'height': 336, 'width': 500, 'date_captured': '2013-11-14 12:36:29', 'flickr_url': 'http://farm3.staticflickr.com/2169/2118578392_1193aa04a0_z.jpg', 'id': 184613}

Example captions:
{'image_id': 149, 'id': 481005, 'caption': 'People gathered outside in a big field on a cloudy day'}
{'image_id': 149, 'id': 481854, 'caption': 'People are flying kites on a high school field.'}
{'image_id': 149, 'id': 483387, 'caption': 'People flying kites in a field on a cloudy day.'}

Caption annotation inspection completed.


In [10]:
# ==================================================
# 10. INSPECT ORIGINAL CAPTION EXAMPLES
# ==================================================

NUM_EXAMPLES = 10

print(f"Displaying {NUM_EXAMPLES} original training captions:\n")

for i, annotation in enumerate(
    captions_data["annotations"][:NUM_EXAMPLES],
    start=1
):
    print(f"{i}. {annotation['caption']!r}")

Displaying 10 original training captions:

1. 'People gathered outside in a big field on a cloudy day'
2. 'People are flying kites on a high school field.'
3. 'People flying kites in a field on a cloudy day.'
4. 'A host of kites being flown on a field by people.'
5. 'a bunch of people are at a grass field '
6. 'There is a small bus with several people standing next to it.'
7. 'people standing besides a bus taking to each other'
8. 'People eating from food trucks near a commemorative archway'
9. 'Several people walking on a sidewalk near a large arch with figures on it.'
10. 'A very big pretty arch way with a bunch of people near it.'


In [11]:
# ==================================================
# 11. INSPECT CAPTION TEXT QUALITY
# ==================================================

# Extract original captions
captions = [
    annotation["caption"]
    for annotation in captions_data["annotations"]
]

# Calculate text-quality statistics
caption_quality_results = {
    "total_captions": len(captions),

    "empty_captions": sum(
        not caption.strip()
        for caption in captions
    ),

    "leading_trailing_whitespace": sum(
        caption != caption.strip()
        for caption in captions
    ),

    "repeated_whitespace": sum(
        caption != " ".join(caption.split())
        for caption in captions
    ),

    "unique_caption_strings": len(set(captions))
}

# Display results
print("Total captions:", caption_quality_results["total_captions"])
print("Empty captions:", caption_quality_results["empty_captions"])

print(
    "Captions with leading/trailing whitespace:",
    caption_quality_results["leading_trailing_whitespace"]
)

print(
    "Captions with repeated whitespace:",
    caption_quality_results["repeated_whitespace"]
)

print(
    "Unique caption strings:",
    caption_quality_results["unique_caption_strings"]
)

Total captions: 49500
Empty captions: 0
Captions with leading/trailing whitespace: 7072
Captions with repeated whitespace: 7555
Unique caption strings: 48996


In [13]:
# ==================================================
# 12. TEST TEXT PREPROCESSING FUNCTION
# ==================================================

test_cases = [
    "  Hello  world  ",
    "Hello\tworld",
    "Hello\nworld",
    "",
    "   ",
    None
]

for caption in test_cases:

    print("Original:", repr(caption))

    try:
        cleaned = clean_caption(caption)
        print("Cleaned: ", repr(cleaned))

    except TypeError as error:
        print("Expected error:", error)

    print("-" * 50)

Original: '  Hello  world  '
Cleaned:  'Hello world'
--------------------------------------------------
Original: 'Hello\tworld'
Cleaned:  'Hello world'
--------------------------------------------------
Original: 'Hello\nworld'
Cleaned:  'Hello world'
--------------------------------------------------
Original: ''
Cleaned:  ''
--------------------------------------------------
Original: '   '
Cleaned:  ''
--------------------------------------------------
Original: None
Expected error: Caption must be a string.
--------------------------------------------------


In [14]:
# ==================================================
# 13. CLEAN CAPTIONS ACROSS ALL DATASET SPLITS
# ==================================================

cleaned_captions_by_split = {}
cleaned_annotations_by_split = {}
caption_cleaning_results = {}

for split in SPLITS:

    print(f"\n{'=' * 45}")
    print(f"CLEANING {split.upper()} CAPTIONS")
    print(f"{'=' * 45}")

    # Load the original annotations for this split
    captions_path = (
        DATA_ROOT / split / "annotations" / "captions.json"
    )

    with open(captions_path, "r", encoding="utf-8") as f:
        original_data = json.load(f)

    original_annotations = original_data["annotations"]

    # Create cleaned copies without modifying original annotations
    cleaned_annotations = []

    for annotation in original_annotations:

        cleaned_annotation = annotation.copy()

        cleaned_annotation["caption"] = clean_caption(
            annotation["caption"]
        )

        cleaned_annotations.append(cleaned_annotation)

    # Extract cleaned caption text
    cleaned_captions = [
        annotation["caption"]
        for annotation in cleaned_annotations
    ]

    # Calculate cleaning statistics
    results = {
        "original_captions": len(original_annotations),
        "cleaned_captions": len(cleaned_captions),

        "empty_cleaned_captions": sum(
            not caption
            for caption in cleaned_captions
        ),

        "remaining_leading_trailing_whitespace": sum(
            caption != caption.strip()
            for caption in cleaned_captions
        ),

        "remaining_repeated_whitespace": sum(
            caption != " ".join(caption.split())
            for caption in cleaned_captions
        ),

        "unique_cleaned_captions": len(set(cleaned_captions))
    }

    # Store results for later use
    cleaned_captions_by_split[split] = cleaned_captions
    cleaned_annotations_by_split[split] = cleaned_annotations
    caption_cleaning_results[split] = results

    # Display results
    for metric, value in results.items():
        print(f"{metric.replace('_', ' ').title()}: {value}")

    # Validate annotation preservation
    assert len(cleaned_annotations) == len(original_annotations)

    assert all(
        original["id"] == cleaned["id"]
        and original["image_id"] == cleaned["image_id"]
        for original, cleaned in zip(
            original_annotations,
            cleaned_annotations
        )
    )

print("\nAll caption cleaning checks completed.")


CLEANING TRAINING CAPTIONS
Original Captions: 49500
Cleaned Captions: 49500
Empty Cleaned Captions: 0
Remaining Leading Trailing Whitespace: 0
Remaining Repeated Whitespace: 0
Unique Cleaned Captions: 48949

CLEANING VALIDATION CAPTIONS
Original Captions: 500
Cleaned Captions: 500
Empty Cleaned Captions: 0
Remaining Leading Trailing Whitespace: 0
Remaining Repeated Whitespace: 0
Unique Cleaned Captions: 499

CLEANING TEST CAPTIONS
Original Captions: 500
Cleaned Captions: 500
Empty Cleaned Captions: 0
Remaining Leading Trailing Whitespace: 0
Remaining Repeated Whitespace: 0
Unique Cleaned Captions: 499

All caption cleaning checks completed.


In [15]:
# ==================================================
# 14. SAVE CLEANED CAPTION ANNOTATIONS
# ==================================================

for split in SPLITS:

    annotations_dir = DATA_ROOT / split / "annotations"
    captions_dir = DATA_ROOT / split / "captions"

    # Create output directory
    captions_dir.mkdir(parents=True, exist_ok=True)

    input_path = annotations_dir / "captions.json"
    output_path = captions_dir / "captions_cleaned.json"

    # Load original annotation data to preserve metadata
    with open(input_path, "r", encoding="utf-8") as f:
        original_data = json.load(f)

    # Reuse the cleaned annotations from the previous cell
    cleaned_annotations = cleaned_annotations_by_split[split]

    # Create a copy of the original structure
    cleaned_data = original_data.copy()
    cleaned_data["annotations"] = cleaned_annotations

    # Validate annotation count and relationships
    assert len(cleaned_annotations) == len(original_data["annotations"])

    assert all(
        original["id"] == cleaned["id"]
        and original["image_id"] == cleaned["image_id"]
        for original, cleaned in zip(
            original_data["annotations"],
            cleaned_annotations
        )
    )

    # Save cleaned captions separately
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            cleaned_data,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"\n{split.upper()}")
    print("Original captions:", len(original_data["annotations"]))
    print("Cleaned captions:", len(cleaned_annotations))
    print("Saved to:", output_path)

print("\nAll cleaned caption files saved successfully.")


TRAINING
Original captions: 49500
Cleaned captions: 49500
Saved to: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\captions\captions_cleaned.json

VALIDATION
Original captions: 500
Cleaned captions: 500
Saved to: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\captions\captions_cleaned.json

TEST
Original captions: 500
Cleaned captions: 500
Saved to: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\captions\captions_cleaned.json

All cleaned caption files saved successfully.


In [16]:
# ==================================================
# 15. VALIDATE SAVED CLEANED CAPTION ANNOTATIONS
# ==================================================

from data.preprocessing.text_preprocessing import clean_caption

for split in SPLITS:

    annotations_dir = DATA_ROOT / split / "annotations"
    captions_dir = DATA_ROOT / split / "captions"

    original_path = annotations_dir / "captions.json"
    cleaned_path = captions_dir / "captions_cleaned.json"

    # Load original and cleaned files
    with open(original_path, "r", encoding="utf-8") as f:
        original_data = json.load(f)

    with open(cleaned_path, "r", encoding="utf-8") as f:
        cleaned_data = json.load(f)

    original_annotations = original_data["annotations"]
    cleaned_annotations = cleaned_data["annotations"]

    # Check annotation counts
    assert len(cleaned_annotations) == len(original_annotations)

    # Check that annotation IDs and image IDs are preserved
    assert [
        (ann["id"], ann["image_id"])
        for ann in original_annotations
    ] == [
        (ann["id"], ann["image_id"])
        for ann in cleaned_annotations
    ]

    # Check that every saved caption is correctly cleaned
    assert all(
        ann["caption"] == clean_caption(ann["caption"])
        for ann in cleaned_annotations
    )

    # Check for empty captions
    empty_captions = sum(
        not ann["caption"].strip()
        for ann in cleaned_annotations
    )

    assert empty_captions == 0

    print(f"\n{split.upper()}")
    print("Saved captions:", len(cleaned_annotations))
    print("Empty captions:", empty_captions)
    print("Annotation IDs preserved: Yes")
    print("Image IDs preserved: Yes")
    print("Cleaning validation: Passed")

print("\nAll saved cleaned caption files passed validation.")


TRAINING
Saved captions: 49500
Empty captions: 0
Annotation IDs preserved: Yes
Image IDs preserved: Yes
Cleaning validation: Passed

VALIDATION
Saved captions: 500
Empty captions: 0
Annotation IDs preserved: Yes
Image IDs preserved: Yes
Cleaning validation: Passed

TEST
Saved captions: 500
Empty captions: 0
Annotation IDs preserved: Yes
Image IDs preserved: Yes
Cleaning validation: Passed

All saved cleaned caption files passed validation.


In [17]:
# ==================================================
# 16. ANALYSE CLEANED CAPTION LENGTHS
# ==================================================

caption_length_results = {}

for split in SPLITS:

    cleaned_path = (
        DATA_ROOT / split / "captions" / "captions_cleaned.json"
    )

    with open(cleaned_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    captions = [
        ann["caption"]
        for ann in data["annotations"]
    ]

    # Calculate caption lengths in words
    lengths = [
        len(caption.split())
        for caption in captions
    ]

    results = {
        "total_captions": len(captions),
        "minimum_words": min(lengths),
        "maximum_words": max(lengths),
        "average_words": sum(lengths) / len(lengths),
        "median_words": sorted(lengths)[len(lengths) // 2],
    }

    caption_length_results[split] = results

    print(f"\n{split.upper()}")
    print("-" * 40)

    for metric, value in results.items():
        if isinstance(value, float):
            print(f"{metric}: {value:.2f}")
        else:
            print(f"{metric}: {value}")

print("\nCaption length analysis completed.")


TRAINING
----------------------------------------
total_captions: 49500
minimum_words: 6
maximum_words: 44
average_words: 10.46
median_words: 10

VALIDATION
----------------------------------------
total_captions: 500
minimum_words: 7
maximum_words: 41
average_words: 10.46
median_words: 10

TEST
----------------------------------------
total_captions: 500
minimum_words: 7
maximum_words: 28
average_words: 10.41
median_words: 10

Caption length analysis completed.


In [18]:
# ==================================================
# 17. CAPTION LENGTH DISTRIBUTION
# ==================================================

length_ranges = {
    "1-5 words": (1, 5),
    "6-10 words": (6, 10),
    "11-15 words": (11, 15),
    "16-20 words": (16, 20),
    "21-30 words": (21, 30),
    "31+ words": (31, float("inf")),
}

caption_length_distribution = {}

for split in SPLITS:

    cleaned_path = (
        DATA_ROOT / split / "captions" / "captions_cleaned.json"
    )

    with open(cleaned_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    lengths = [
        len(ann["caption"].split())
        for ann in data["annotations"]
    ]

    distribution = {}

    for label, (lower, upper) in length_ranges.items():
        count = sum(
            lower <= length <= upper
            for length in lengths
        )

        distribution[label] = {
            "count": count,
            "percentage": count / len(lengths) * 100
        }

    caption_length_distribution[split] = distribution

    print(f"\n{split.upper()}")
    print("-" * 45)

    for label, values in distribution.items():
        print(
            f"{label:<15} "
            f"{values['count']:>6} captions "
            f"({values['percentage']:.2f}%)"
        )

print("\nCaption length distribution completed.")


TRAINING
---------------------------------------------
1-5 words            0 captions (0.00%)
6-10 words       29990 captions (60.59%)
11-15 words      17812 captions (35.98%)
16-20 words       1418 captions (2.86%)
21-30 words        249 captions (0.50%)
31+ words           31 captions (0.06%)

VALIDATION
---------------------------------------------
1-5 words            0 captions (0.00%)
6-10 words         308 captions (61.60%)
11-15 words        180 captions (36.00%)
16-20 words          8 captions (1.60%)
21-30 words          3 captions (0.60%)
31+ words            1 captions (0.20%)

TEST
---------------------------------------------
1-5 words            0 captions (0.00%)
6-10 words         316 captions (63.20%)
11-15 words        167 captions (33.40%)
16-20 words         14 captions (2.80%)
21-30 words          3 captions (0.60%)
31+ words            0 captions (0.00%)

Caption length distribution completed.


In [19]:
# ==================================================
# 18. SAVE TEXT ANALYSIS REPORT
# ==================================================

text_analysis_report = {
    "caption_length_statistics": caption_length_results,
    "caption_length_distribution": caption_length_distribution,
    "observations": [
        "Most captions contain between 6 and 15 words.",
        "Caption length distributions are similar across the three splits.",
        "Captions longer than 20 words are uncommon.",
        "No captions shorter than 6 words were found."
    ]
}

report_path = DATA_ROOT / "text_analysis_report.json"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(
        text_analysis_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Text analysis report saved successfully.")
print("Location:", report_path)

Text analysis report saved successfully.
Location: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\text_analysis_report.json


In [20]:
# ==================================================
# 19. CAPTION VOCABULARY ANALYSIS
# ==================================================

import re
from collections import Counter

caption_vocabulary_results = {}

for split in SPLITS:

    cleaned_path = (
        DATA_ROOT / split / "captions" / "captions_cleaned.json"
    )

    with open(cleaned_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    captions = [
        ann["caption"]
        for ann in data["annotations"]
    ]

    # Tokenize captions into lowercase words
    words = []

    for caption in captions:
        tokens = re.findall(r"\b[a-zA-Z]+\b", caption.lower())
        words.extend(tokens)

    word_counts = Counter(words)

    results = {
        "total_words": len(words),
        "unique_words": len(word_counts),
        "most_common_20": word_counts.most_common(20)
    }

    caption_vocabulary_results[split] = results

    print(f"\n{split.upper()}")
    print("-" * 50)
    print("Total words:", results["total_words"])
    print("Unique words:", results["unique_words"])
    print("\nMost frequent 20 words:")

    for word, count in results["most_common_20"]:
        print(f"{word:<15} {count}")

print("\nCaption vocabulary analysis completed.")


TRAINING
--------------------------------------------------
Total words: 518774
Unique words: 9514

Most frequent 20 words:
a               81950
on              17954
of              17051
the             16502
in              15218
with            12976
and             11998
is              8400
man             6082
to              5784
sitting         4457
an              4317
two             4117
people          3547
at              3498
standing        3423
are             3421
next            3158
white           2999
woman           2823

VALIDATION
--------------------------------------------------
Total words: 5235
Unique words: 950

Most frequent 20 words:
a               819
on              232
the             171
of              164
in              154
with            128
and             119
is              89
sitting         58
to              50
man             49
two             44
an              38
table           38
at              37
people          37
street       

In [21]:
# ==================================================
# 20. SAVE VOCABULARY ANALYSIS REPORT
# ==================================================

vocabulary_report = {
    "vocabulary_statistics": {
        split: {
            "total_words": results["total_words"],
            "unique_words": results["unique_words"]
        }
        for split, results in caption_vocabulary_results.items()
    },
    "most_common_words": {
        split: [
            {"word": word, "count": count}
            for word, count in results["most_common_20"]
        ]
        for split, results in caption_vocabulary_results.items()
    },
    "method": {
        "case": "lowercase",
        "tokenization": "alphabetic tokens using regular expressions",
        "stopword_removal": False,
        "stemming": False,
        "lemmatization": False
    }
}

vocabulary_report_path = DATA_ROOT / "vocabulary_analysis_report.json"

with open(vocabulary_report_path, "w", encoding="utf-8") as f:
    json.dump(
        vocabulary_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Vocabulary analysis report saved successfully.")
print("Location:", vocabulary_report_path)

Vocabulary analysis report saved successfully.
Location: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\vocabulary_analysis_report.json


In [22]:
print("CLIP image processor configuration:")
print(clip_processor.image_processor)

CLIP image processor configuration:
CLIPImageProcessorPil {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "CLIPImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}



In [24]:
# ==================================================
# 21. GENERATE IMAGE, CAPTION AND PREPROCESSING METADATA
# ==================================================

import statistics
from PIL import Image

for split in SPLITS:

    split_dir = DATA_ROOT / split
    images_dir = split_dir / "images"
    annotations_dir = split_dir / "annotations"
    captions_dir = split_dir / "captions"
    metadata_dir = split_dir / "metadata"

    metadata_dir.mkdir(parents=True, exist_ok=True)

    # Load original COCO image records
    with open(annotations_dir / "captions.json", "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    # Load cleaned captions
    with open(captions_dir / "captions_cleaned.json", "r", encoding="utf-8") as f:
        cleaned_data = json.load(f)

    image_records = coco_data["images"]
    caption_annotations = cleaned_data["annotations"]

    # Validate expected counts
    assert len(image_records) == SPLIT_SIZES[split]
    assert len(caption_annotations) == SPLIT_SIZES[split] * 5

    # ----------------------------------
    # 1. IMAGE METADATA
    # ----------------------------------

    image_metadata = []

    for image_record in image_records:

        filename = image_record["file_name"]
        image_path = images_dir / filename

        assert image_path.exists(), f"Missing image: {image_path}"

        with Image.open(image_path) as img:

            image_metadata.append({
                "image_id": image_record["id"],
                "file_name": filename,
                "original_width": img.width,
                "original_height": img.height,
                "original_mode": img.mode,
                "format": img.format,
                "file_size_bytes": image_path.stat().st_size,
                "original_exif_orientation": img.getexif().get(274, 1)
            })

    image_metadata_output = {
        "split": split,
        "image_count": len(image_metadata),
        "images": image_metadata
    }

    with open(metadata_dir / "image_metadata.json", "w", encoding="utf-8") as f:
        json.dump(image_metadata_output, f, ensure_ascii=False, indent=2)

    # ----------------------------------
    # 2. CAPTION METADATA
    # ----------------------------------

    captions = [
        annotation["caption"]
        for annotation in caption_annotations
    ]

    word_counts = [
        len(caption.split())
        for caption in captions
    ]

    caption_metadata = {
        "split": split,
        "caption_count": len(captions),
        "unique_caption_count": len(set(captions)),
        "empty_caption_count": sum(not caption for caption in captions),
        "word_statistics": {
            "average": statistics.mean(word_counts) if word_counts else 0,
            "median": statistics.median(word_counts) if word_counts else 0,
            "minimum": min(word_counts) if word_counts else 0,
            "maximum": max(word_counts) if word_counts else 0
        },
        "cleaning_operations": [
            "Normalize consecutive whitespace",
            "Remove leading and trailing whitespace"
        ]
    }

    with open(metadata_dir / "caption_metadata.json", "w", encoding="utf-8") as f:
        json.dump(caption_metadata, f, ensure_ascii=False, indent=2)

    # ----------------------------------
    # 3. PREPROCESSING METADATA
    # ----------------------------------

    crop_size = clip_processor.image_processor.crop_size

    preprocessing_metadata = {
        "split": split,
        "image_preprocessing": {
            "exif_orientation_correction": True,
            "colour_conversion": "RGB",
            "image_processor": "CLIP image processor",
            "model": CLIP_MODEL_NAME,
            "output_tensor_shape": [
                1, 3,
                crop_size["height"],
                crop_size["width"]
            ],
            "output_dtype": "float32",
            "operations": [
                "EXIF orientation correction",
                "RGB conversion",
                "Model-specific resizing",
                "Center cropping",
                "Pixel normalization"
            ]
        },
        "caption_preprocessing": {
            "cleaning_function": "clean_caption",
            "operations": [
                "Whitespace normalization",
                "Leading and trailing whitespace removal"
            ],
            "preserve_original_captions": True,
            "text_embedding_model": SBERT_MODEL_NAME
        }
    }

    with open(metadata_dir / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
        json.dump(
            preprocessing_metadata,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"\n{split.upper()}")
    print("Images recorded:", len(image_metadata))
    print("Captions recorded:", len(captions))
    print("Metadata files saved:", 3)
    print("Metadata directory:", metadata_dir)

print("\nAll metadata files regenerated successfully.")


TRAINING
Images recorded: 9900
Captions recorded: 49500
Metadata files saved: 3
Metadata directory: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\metadata

VALIDATION
Images recorded: 100
Captions recorded: 500
Metadata files saved: 3
Metadata directory: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\metadata

TEST
Images recorded: 100
Captions recorded: 500
Metadata files saved: 3
Metadata directory: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\metadata

All metadata files regenerated successfully.


In [25]:
# ==================================================
# 22. VALIDATE GENERATED METADATA FILES
# ==================================================

metadata_files = [
    "image_metadata.json",
    "caption_metadata.json",
    "preprocessing_metadata.json"
]

for split in SPLITS:

    metadata_dir = DATA_ROOT / split / "metadata"

    print(f"\n{split.upper()}")
    print("-" * 45)

    for filename in metadata_files:

        file_path = metadata_dir / filename

        assert file_path.exists(), f"Missing file: {file_path}"

        with open(file_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        assert metadata["split"] == split

        if filename == "image_metadata.json":
            assert metadata["image_count"] == SPLIT_SIZES[split]
            assert len(metadata["images"]) == SPLIT_SIZES[split]

        elif filename == "caption_metadata.json":
            expected_captions = SPLIT_SIZES[split] * 5
            assert metadata["caption_count"] == expected_captions
            assert metadata["empty_caption_count"] == 0

        elif filename == "preprocessing_metadata.json":
            assert metadata["image_preprocessing"]["model"] == CLIP_MODEL_NAME
            assert (
                metadata["caption_preprocessing"]["text_embedding_model"]
                == SBERT_MODEL_NAME
            )

        print(f"{filename}: Passed")

print("\nAll metadata files passed validation.")


TRAINING
---------------------------------------------
image_metadata.json: Passed
caption_metadata.json: Passed
preprocessing_metadata.json: Passed

VALIDATION
---------------------------------------------
image_metadata.json: Passed
caption_metadata.json: Passed
preprocessing_metadata.json: Passed

TEST
---------------------------------------------
image_metadata.json: Passed
caption_metadata.json: Passed
preprocessing_metadata.json: Passed

All metadata files passed validation.


In [26]:
%pip install -U sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import sentence_transformers

print("Sentence-Transformers version:",
      sentence_transformers.__version__)

Sentence-Transformers version: 6.2.0


In [28]:
# ==================================================
# 23. RECORD SOFTWARE AND MODEL REPRODUCIBILITY
# ==================================================

import platform
import transformers
import PIL
import sentence_transformers
import torch

from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# 1. Load Sentence-BERT using central configuration
# --------------------------------------------------

sbert_model = SentenceTransformer(SBERT_MODEL_NAME)


# --------------------------------------------------
# 2. Record software versions
# --------------------------------------------------

software_versions = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "pillow": PIL.__version__,
    "sentence_transformers": sentence_transformers.__version__,
}


# --------------------------------------------------
# 3. Record CLIP configuration
# --------------------------------------------------

clip_configuration = {
    "model_name": CLIP_MODEL_NAME,
    "processor_configuration": clip_processor.to_dict(),
    "image_processor_configuration":
        clip_processor.image_processor.to_dict(),
    "tokenizer_configuration":
        clip_processor.tokenizer.init_kwargs,
}


# --------------------------------------------------
# 4. Record Sentence-BERT configuration
# --------------------------------------------------

sbert_transformer = sbert_model[0]

sbert_configuration = {
    "model_name": SBERT_MODEL_NAME,
    "max_sequence_length": sbert_model.max_seq_length,
    "embedding_dimension":
        sbert_model.get_sentence_embedding_dimension(),
    "similarity_function": str(sbert_model.similarity_fn_name),
    "transformer_model_configuration":
        sbert_transformer.auto_model.config.to_dict(),
    "tokenizer_configuration":
        sbert_transformer.tokenizer.init_kwargs,
    "modules": [
        {
            "name": name,
            "type": type(module).__name__
        }
        for name, module in sbert_model._modules.items()
    ],
}


# --------------------------------------------------
# 5. Combine reproducibility information
# --------------------------------------------------

reproducibility_metadata = {
    "software_versions": software_versions,
    "models": {
        "clip": clip_configuration,
        "sentence_bert": sbert_configuration,
    },
}


# --------------------------------------------------
# 6. Update preprocessing metadata for all splits
# --------------------------------------------------

for split in SPLITS:

    metadata_path = (
        DATA_ROOT / split / "metadata" / "preprocessing_metadata.json"
    )

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    metadata["reproducibility"] = reproducibility_metadata

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(
            metadata,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )

    print(f"Updated: {metadata_path}")


# --------------------------------------------------
# 7. Display reproducibility information
# --------------------------------------------------

print("\nSoftware versions:")
print(json.dumps(software_versions, indent=2))

print(
    "\nCLIP image processor:",
    CLIP_MODEL_NAME
)

print(
    "\nSentence-BERT embedding dimension:",
    sbert_model.get_sentence_embedding_dimension()
)

print("\nReproducibility metadata successfully recorded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Updated: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\training\metadata\preprocessing_metadata.json
Updated: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\validation\metadata\preprocessing_metadata.json
Updated: C:\Users\iso99\Desktop\Ahmed\current\Image-Captioning-RAG\02_processed_data\test\metadata\preprocessing_metadata.json

Software versions:
{
  "python": "3.11.4",
  "pytorch": "2.15.0+cpu",
  "transformers": "5.22.1",
  "pillow": "10.3.0",
  "sentence_transformers": "6.2.0"
}

CLIP image processor: openai/clip-vit-base-patch32

Sentence-BERT embedding dimension: 384

Reproducibility metadata successfully recorded.


C:\Users\iso99\AppData\Local\Temp\ipykernel_25316\841291167.py:58: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sbert_model.get_sentence_embedding_dimension(),
C:\Users\iso99\AppData\Local\Temp\ipykernel_25316\841291167.py:128: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sbert_model.get_sentence_embedding_dimension()


In [29]:
# ==================================================
# 10. INSPECT ORIGINAL CAPTION EXAMPLES
# ==================================================

NUM_EXAMPLES = 10

print(f"Displaying {NUM_EXAMPLES} original training captions:\n")

for i, annotation in enumerate(
    captions_data["annotations"][:NUM_EXAMPLES],
    start=1
):
    print(f"{i}. {annotation['caption']!r}")

Displaying 10 original training captions:

1. 'People gathered outside in a big field on a cloudy day'
2. 'People are flying kites on a high school field.'
3. 'People flying kites in a field on a cloudy day.'
4. 'A host of kites being flown on a field by people.'
5. 'a bunch of people are at a grass field '
6. 'There is a small bus with several people standing next to it.'
7. 'people standing besides a bus taking to each other'
8. 'People eating from food trucks near a commemorative archway'
9. 'Several people walking on a sidewalk near a large arch with figures on it.'
10. 'A very big pretty arch way with a bunch of people near it.'


In [30]:

for split in ["training", "validation", "test"]:

    metadata_path = (
        DATA_ROOT / split / "metadata" / "preprocessing_metadata.json"
    )

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    reproducibility = metadata["reproducibility"]

    print(f"\n--- {split.upper()} ---")
    print("Python:", reproducibility["software_versions"]["python"])
    print("PyTorch:", reproducibility["software_versions"]["pytorch"])
    print("Transformers:", reproducibility["software_versions"]["transformers"])
    print("Pillow:", reproducibility["software_versions"]["pillow"])
    print("Sentence-Transformers:",
          reproducibility["software_versions"]["sentence_transformers"])

    print("CLIP:", reproducibility["models"]["clip"]["model_name"])
    print("Sentence-BERT:",
          reproducibility["models"]["sentence_bert"]["model_name"])
    print("SBERT dimension:",
          reproducibility["models"]["sentence_bert"]["embedding_dimension"])



--- TRAINING ---
Python: 3.11.4
PyTorch: 2.15.0+cpu
Transformers: 5.22.1
Pillow: 10.3.0
Sentence-Transformers: 6.2.0
CLIP: openai/clip-vit-base-patch32
Sentence-BERT: sentence-transformers/all-MiniLM-L6-v2
SBERT dimension: 384

--- VALIDATION ---
Python: 3.11.4
PyTorch: 2.15.0+cpu
Transformers: 5.22.1
Pillow: 10.3.0
Sentence-Transformers: 6.2.0
CLIP: openai/clip-vit-base-patch32
Sentence-BERT: sentence-transformers/all-MiniLM-L6-v2
SBERT dimension: 384

--- TEST ---
Python: 3.11.4
PyTorch: 2.15.0+cpu
Transformers: 5.22.1
Pillow: 10.3.0
Sentence-Transformers: 6.2.0
CLIP: openai/clip-vit-base-patch32
Sentence-BERT: sentence-transformers/all-MiniLM-L6-v2
SBERT dimension: 384
